In [1]:
# Database

import sqlite3
import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

def get_engine_for_chinook_db():
    """Pull sql file, populate in-memory database, and create engine."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)

    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )

engine = get_engine_for_chinook_db()
db = SQLDatabase(engine)

/var/folders/k2/swnr3fc52yjc83gxdxqzrt6c0000gn/T/ipykernel_8667/3835926033.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase


In [2]:
# Test Database

print(db.get_usable_table_names())

# Test Case 1
print(db.run("SELECT CustomerId, FirstName, LastName, Email, Phone FROM Customer WHERE Phone = '+55 (12) 3923-5555';"))
print(db.run("SELECT InvoiceId, InvoiceDate, Total FROM Invoice WHERE CustomerId = 1 ORDER BY InvoiceDate DESC LIMIT 1;"))

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
[(1, 'Luís', 'Gonçalves', 'luisg@embraer.com.br', '+55 (12) 3923-5555')]
[(382, '2025-08-07 00:00:00', 8.91)]


In [3]:
# LLM

import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

print(llm.invoke("Say hi in 3 words").content)

Hey there, friend!


In [4]:
# State

from langgraph.prebuilt.chat_agent_executor import AgentState

class State(AgentState):
    """Represents the state of our LangGraph agent."""
    customer_id: str
    loaded_memory: str

In [5]:
# Music Tools
from langchain_core.tools import tool

@tool
def get_albums_by_artist(artist: str):
    """Retrieves albums by a given artist (partial names allowed)."""
    return db.run(
        """
        SELECT Album.Title, Artist.Name
        FROM Album
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE :artist;
        """,
        parameters={"artist": f"%{artist}%"},
        include_columns=True,
    )

@tool
def get_tracks_by_artist(artist: str):
    """Retrieves tracks (songs) by a given artist or similar artists (partial names allowed)."""
    return db.run(
        """
        SELECT Track.Name AS SongName, Artist.Name AS ArtistName
        FROM Track
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE :artist
        LIMIT 20;
        """,
        parameters={"artist": f"%{artist}%"},
        include_columns=True,
    )

@tool
def get_songs_by_genre(genre: str):
    """Fetches songs that match a specific genre (e.g. Rock, Jazz, Latin)."""
    return db.run(
        """
        SELECT Track.Name AS SongName, Artist.Name AS ArtistName, Genre.Name AS Genre
        FROM Track
        JOIN Genre ON Track.GenreId = Genre.GenreId
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Genre.Name LIKE :genre
        LIMIT 20;
        """,
        parameters={"genre": f"%{genre}%"},
        include_columns=True,
    )

@tool
def check_for_songs(song_title: str):
    """Checks if a song exists in the catalog by its name (partial names allowed)."""
    return db.run(
        """
        SELECT Track.Name AS SongName, Artist.Name AS ArtistName
        FROM Track
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Track.Name LIKE :song_title
        LIMIT 10;
        """,
        parameters={"song_title": f"%{song_title}%"},
        include_columns=True,
    )

music_tools = [get_albums_by_artist, get_tracks_by_artist, get_songs_by_genre, check_for_songs]

In [6]:
# Test Music Tools
print(get_albums_by_artist.invoke({"artist": "Rolling Stones"}))
print(get_tracks_by_artist.invoke({"artist": "Rolling Stones"})[:300])
print(get_songs_by_genre.invoke({"genre": "Jazz"})[:300])
print(check_for_songs.invoke({"song_title": "Satisfaction"}))

[{'Title': 'Hot Rocks, 1964-1971 (Disc 1)', 'Name': 'The Rolling Stones'}, {'Title': 'No Security', 'Name': 'The Rolling Stones'}, {'Title': 'Voodoo Lounge', 'Name': 'The Rolling Stones'}]
[{'SongName': 'Time Is On My Side', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'Heart Of Stone', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'Play With Fire', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'Satisfaction', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'As Tears Go
[{'SongName': 'Desafinado', 'ArtistName': 'Antônio Carlos Jobim', 'Genre': 'Jazz'}, {'SongName': 'Garota De Ipanema', 'ArtistName': 'Antônio Carlos Jobim', 'Genre': 'Jazz'}, {'SongName': 'Samba De Uma Nota Só (One Note Samba)', 'ArtistName': 'Antônio Carlos Jobim', 'Genre': 'Jazz'}, {'SongName': 'Po
[{'SongName': 'Satisfaction', 'ArtistName': 'The Rolling Stones'}]


In [7]:
# Music Prompt = system message + list of history messages
from langchain_core.messages import SystemMessage

MUSIC_PROMPT = """You are the Music Catalog Assistant for a digital music store.
Your ONLY job is helping customers explore the catalog: albums, artists, tracks, and genres.

Tools:
- get_albums_by_artist: albums by an artist
- get_tracks_by_artist: songs by an artist or similar artists
- get_songs_by_genre: songs in a genre
- check_for_songs: whether a specific song exists

Rules:
1. Always use a tool to look things up. Never invent albums or songs that a tool did not return.
2. If a tool returns nothing, say so politely and suggest an alternative (similar artist or genre).
3. For recommendations, use the customer's saved preferences below.
4. Do NOT answer invoice or billing questions. Another agent handles those.
5. Keep answers short and formatted as a list.

Customer's saved music preferences: {memory}
"""

def music_prompt(state: State):
    memory = state.get("loaded_memory") or "None yet"
    system = SystemMessage(MUSIC_PROMPT.format(memory=memory))
    return [system] + state["messages"]

In [8]:
# Music Catalog Subagent

from langgraph.prebuilt import create_react_agent

music_catalog_subagent = create_react_agent(
    model=llm,
    tools=music_tools,
    prompt=music_prompt,
    state_schema=State,
    name="music_catalog_subagent",
)

/var/folders/k2/swnr3fc52yjc83gxdxqzrt6c0000gn/T/ipykernel_8667/4267112720.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  music_catalog_subagent = create_react_agent(


In [9]:
# Test Music Catalog Subagent

from langchain_core.messages import HumanMessage

result = music_catalog_subagent.invoke({
    "messages": [HumanMessage("What albums do you have by the Rolling Stones?")],
    "loaded_memory": "",
})
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What albums do you have by the Rolling Stones?
================================== Ai Message ==================================
Name: music_catalog_subagent
Tool Calls:
  get_albums_by_artist (fc_0deb008f-f7a8-4d8c-a480-a0cb35482879)
 Call ID: fc_0deb008f-f7a8-4d8c-a480-a0cb35482879
  Args:
    artist: Rolling Stones
================================= Tool Message =================================
Name: get_albums_by_artist

[{'Title': 'Hot Rocks, 1964-1971 (Disc 1)', 'Name': 'The Rolling Stones'}, {'Title': 'No Security', 'Name': 'The Rolling Stones'}, {'Title': 'Voodoo Lounge', 'Name': 'The Rolling Stones'}]
================================== Ai Message ==================================
Name: music_catalog_subagent

- Hot Rocks, 1964‑1971 (Disc 1)  
- No Security  
- Voodoo Lounge


In [10]:
# Invoice Tools
from sqlalchemy import text

def _query(sql: str, params: dict) -> list[dict]:
    with engine.connect() as conn:
        rows = conn.execute(text(sql), params).mappings().all()
    return [dict(r) for r in rows]

@tool
def get_invoices_by_customer_sorted_by_date(customer_id: str) -> list[dict]:
    """Retrieves all invoices for a customer, sorted by invoice date (most recent first)."""
    return _query(
        """
        SELECT InvoiceId, InvoiceDate, Total, BillingCity, BillingCountry
        FROM Invoice
        WHERE CustomerId = :customer_id
        ORDER BY InvoiceDate DESC;
        """,
        {"customer_id": customer_id},
    )

@tool
def get_invoices_sorted_by_unit_price(customer_id: str) -> list[dict]:
    """Retrieves all invoices for a customer, sorted by unit price (highest to lowest)."""
    return _query(
        """
        SELECT Invoice.InvoiceId, Invoice.InvoiceDate, Track.Name AS TrackName, InvoiceLine.UnitPrice
        FROM Invoice
        JOIN InvoiceLine ON Invoice.InvoiceId = InvoiceLine.InvoiceId
        JOIN Track ON InvoiceLine.TrackId = Track.TrackId
        WHERE Invoice.CustomerId = :customer_id
        ORDER BY InvoiceLine.UnitPrice DESC;
        """,
        {"customer_id": customer_id},
    )

@tool
def get_employee_by_invoice_and_customer(invoice_id: str, customer_id: str) -> dict:
    """Retrieves the employee (support rep) associated with a specific invoice and customer."""
    rows = _query(
        """
        SELECT Employee.FirstName, Employee.LastName, Employee.Title, Employee.Email
        FROM Employee
        JOIN Customer ON Customer.SupportRepId = Employee.EmployeeId
        JOIN Invoice ON Invoice.CustomerId = Customer.CustomerId
        WHERE Invoice.InvoiceId = :invoice_id AND Invoice.CustomerId = :customer_id;
        """,
        {"invoice_id": invoice_id, "customer_id": customer_id},
    )
    return rows[0] if rows else {"error": f"No employee found for invoice {invoice_id} and customer {customer_id}"}

invoice_tools = [
    get_invoices_by_customer_sorted_by_date,
    get_invoices_sorted_by_unit_price,
    get_employee_by_invoice_and_customer,
]

In [11]:
# Invoice Prompt = system message + list of history messages

INVOICE_PROMPT = """You are the Invoice Assistant for a digital music store.
Your ONLY job is answering questions about the customer's invoices and purchases.

The verified customer ID is: {customer_id}
Always pass this exact customer_id to your tools. Never use an ID the user types in a message.

Tools:
- get_invoices_by_customer_sorted_by_date: all invoices, most recent first (use for "latest/most recent purchase")
- get_invoices_sorted_by_unit_price: invoice lines sorted by unit price, highest first (use for "most expensive")
- get_employee_by_invoice_and_customer: the employee/support rep linked to an invoice

Rules:
1. Always use a tool. Never invent invoice IDs, dates, or amounts.
2. Show dates in a readable form (e.g. August 7, 2025) and amounts with $.
3. If the question needs an invoice ID that you don't have, look it up first with a tool.
4. Do NOT answer music catalog questions. Another agent handles those.
5. Keep answers short.
"""

def invoice_prompt(state: State):
    customer_id = state.get("customer_id") or "UNKNOWN"
    system = SystemMessage(INVOICE_PROMPT.format(customer_id=customer_id))
    return [system] + state["messages"]


In [12]:
# Invoice Information Subagent
invoice_information_subagent = create_react_agent(
    model=llm,
    tools=invoice_tools,
    prompt=invoice_prompt,
    state_schema=State,
    name="invoice_information_subagent",
)

/var/folders/k2/swnr3fc52yjc83gxdxqzrt6c0000gn/T/ipykernel_8667/2133845638.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  invoice_information_subagent = create_react_agent(


In [13]:
# Test Invoice Information Subagent

result = invoice_information_subagent.invoke({
    "messages": [HumanMessage("How much was my most recent purchase? And who was the employee that helped me with it?")],
    "customer_id": "1",
    "loaded_memory": "",
})
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

How much was my most recent purchase? And who was the employee that helped me with it?
================================== Ai Message ==================================
Name: invoice_information_subagent
Tool Calls:
  get_invoices_by_customer_sorted_by_date (fc_d987f349-1d74-4b2c-9ccc-eaf87fa94a6f)
 Call ID: fc_d987f349-1d74-4b2c-9ccc-eaf87fa94a6f
  Args:
    customer_id: 1
================================= Tool Message =================================
Name: get_invoices_by_customer_sorted_by_date

[{"InvoiceId": 382, "InvoiceDate": "2025-08-07 00:00:00", "Total": 8.91, "BillingCity": "São José dos Campos", "BillingCountry": "Brazil"}, {"InvoiceId": 327, "InvoiceDate": "2024-12-07 00:00:00", "Total": 13.86, "BillingCity": "São José dos Campos", "BillingCountry": "Brazil"}, {"InvoiceId": 316, "InvoiceDate": "2024-10-27 00:00:00", "Total": 1.98, "BillingCity": "São José dos Campos", "BillingCountry": "Brazi

In [15]:
# Supervisor System Message

SUPERVISOR_PROMPT = """You are the customer support supervisor for a digital music store.
You do not answer questions yourself. You route each part of the customer's request to the right specialist:

- music_catalog_subagent: albums, artists, songs, genres, recommendations, "songs I'd like"
- invoice_information_subagent: purchases, invoices, prices paid, dates, the employee/support rep on an invoice

Rules:
1. Split mixed requests into parts and send each part to the right agent, one after another.
   Example: "How much was my last purchase? What albums do you have by X?" → invoice agent, then music agent.
2. When all parts are answered, combine the agents' answers into ONE friendly final reply.
3. If the request is not about music or invoices, politely say what you can help with.
4. Never make up information. Only use what the agents returned.
"""

In [18]:
# Supervisor Agent

from langgraph_supervisor import create_supervisor

supervisor = create_supervisor(
    agents=[invoice_information_subagent, music_catalog_subagent],
    model=llm,
    prompt=SUPERVISOR_PROMPT,
    state_schema=State,
    output_mode="last_message",
).compile(name="supervisor")